# Fig. 4(a)

In [ ]:
import os
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from tqdm import tqdm
from matplotlib.colors import TwoSlopeNorm

responses_dir = "../results/responses/loom/202603/"
summary_dir = "../results/loom/plots_peak_trace_summary_only/"
os.makedirs(summary_dir, exist_ok=True)

cell_type_file = "../data/cell_type.txt"
classification_file = "../data/classification.txt"

GRID_Y = 41
GRID_X = 82
TOP_K = 25

HIGHLIGHT_TYPES = {"DNp01", "DNp02", "DNp03", "DNp04", "DNp11"}

FONT_SIZE = 20  

cell_type_df = pd.read_csv(
    cell_type_file,
    header=None,
    names=["root_id", "primary_type", "additional_type"]
)
cell_type_df["root_id"] = cell_type_df["root_id"].astype(str)
cell_type_df.set_index("root_id", inplace=True)

classification_df = pd.read_csv(
    classification_file,
    dtype={"root_id": str}
)
classification_df["root_id"] = classification_df["root_id"].astype(str)
classification_df.set_index("root_id", inplace=True)

all_files = glob.glob(os.path.join(responses_dir, "*.npz"))
responses_by_ms = {}

print("Loading response files...")
for f in tqdm(all_files):
    base = os.path.basename(f)

    try:
        x = int(base.split("_")[0][1:])
        y = int(base.split("_")[1][1:])
    except Exception:
        continue

    if x >= GRID_X or y >= GRID_Y:
        continue

    ms = base.split("_")[-1].replace(".npz", "")
    data = np.load(f, allow_pickle=True)

    if "neuron_ids" not in data or "responses" not in data:
        continue

    neuron_ids = data["neuron_ids"]
    responses = data["responses"]

    stim_dict = responses_by_ms.setdefault(ms, {}).setdefault((x, y), {})
    for nid, resp in zip(neuron_ids, responses):
        stim_dict[str(nid)] = resp 

top_neurons_by_ms_side = {}

for ms, stim_dict in responses_by_ms.items():
    peak_by_neuron = {}

    for neuron_resp_dict in stim_dict.values():
        for nid, resp in neuron_resp_dict.items():
            peak = np.max(resp)
            if peak > 0:
                peak_by_neuron[nid] = max(
                    peak_by_neuron.get(nid, 0.0),
                    peak
                )

    for side in ["L", "R"]:
        selected = []
        for nid, _ in sorted(
            peak_by_neuron.items(),
            key=lambda x: x[1],
            reverse=True
        ):
            ntype = cell_type_df["primary_type"].get(nid)
            if ntype is None or ntype == "Unknown":
                continue

            nside = classification_df["side"].get(nid, "").upper()
            if not nside.startswith(side):
                continue

            selected.append(nid)
            if len(selected) >= TOP_K:
                break

        top_neurons_by_ms_side[(ms, side)] = selected

def plot_peak_trace_summary(ms, side, stim_dict, neuron_list):
    traces = []
    peaks = []
    type_labels = []
    xy_labels = []

    for nid in neuron_list:
        best_peak = -np.inf
        best_trace = None
        best_xy = None

        for (x, y), neuron_resp_dict in stim_dict.items():
            if nid not in neuron_resp_dict:
                continue

            resp = neuron_resp_dict[nid]
            peak = np.max(resp)

            if peak > best_peak:
                best_peak = peak
                best_trace = resp
                best_xy = (x, y)

        if best_trace is None or best_peak <= 0:
            continue

        ntype = cell_type_df["primary_type"].get(nid)
        if ntype is None or ntype == "Unknown":
            continue

        traces.append(best_trace)
        peaks.append(best_peak)
        type_labels.append(ntype)
        xy_labels.append(f"({best_xy[0]}, {best_xy[1]})")

    if len(traces) == 0:
        return

    traces = np.array(traces)

    order = np.argsort(peaks)[::-1]
    traces = traces[order]
    type_labels = [type_labels[i] for i in order]
    xy_labels = [xy_labels[i] for i in order]

    vmax = np.max(traces)
    norm = TwoSlopeNorm(vmin=-vmax, vcenter=0.0, vmax=vmax)

    fig, ax = plt.subplots(figsize=(6, 11))

    im = ax.imshow(
        traces,
        aspect="auto",
        origin="lower",
        cmap="RdBu_r",
        norm=norm
    )

    ax.set_xlabel("Time", fontsize=FONT_SIZE)
    ax.set_yticks(np.arange(len(type_labels)))
    ax.set_yticklabels(type_labels, fontsize=FONT_SIZE)
    ax.set_title(side, fontsize=FONT_SIZE)

    for tick, label in zip(ax.get_yticklabels(), type_labels):
        if label in HIGHLIGHT_TYPES:
            tick.set_fontweight("bold")

    cbar = fig.colorbar(
        im,
        ax=ax,
        fraction=0.035,
        pad=0.05
    )
    cbar.ax.tick_params(labelsize=FONT_SIZE) 

    fname = f"summary_peak_trace_{side}_{ms}.png"
    fig.savefig(os.path.join(summary_dir, fname), dpi=600, bbox_inches="tight")
    plt.close(fig)

print("Start plotting clean summary heatmaps (safe indexing)...")

for ms, stim_dict in responses_by_ms.items():
    for side in ["L", "R"]:
        plot_peak_trace_summary(
            ms,
            side,
            stim_dict,
            top_neurons_by_ms_side.get((ms, side), [])
        )

print("\n✅ Summary heatmaps finished (fonts enlarged).")


# Fig. 4(b)

In [ ]:
import os
import glob
import gc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm
import scienceplots

plt.style.use(['science', 'nature', 'no-latex'])

plt.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Liberation Sans"],
    "font.size": 32,
    "axes.labelsize": 32,
    "axes.titlesize": 32,
    "xtick.labelsize": 32,
    "ytick.labelsize": 32,
    "axes.linewidth": 1.5,
})

responses_dir = "../results/responses/loom/202603/"
peak_trace_root = "../results/loom/plots_peak_trace_max_rv/"
os.makedirs(peak_trace_root, exist_ok=True)

cell_type_file = "../data/cell_type.txt"
classification_file = "../data/classification.txt"

TARGET_TYPES = ["DNp01", "DNp02", "DNp03", "DNp04", "DNp11"]

cell_type_df = pd.read_csv(
    cell_type_file,
    header=None,
    names=["root_id", "primary_type", "additional_type"]
)
cell_type_df.set_index("root_id", inplace=True)
cell_type_df.index = cell_type_df.index.astype(str)

classification_df = pd.read_csv(
    classification_file,
    dtype={"root_id": str}
)
classification_df.set_index("root_id", inplace=True)
classification_df.index = classification_df.index.astype(str)

all_files = glob.glob(os.path.join(responses_dir, "*.npz"))
responses_by_ms = {}

print("Loading response files...")
for f in tqdm(all_files):
    base = os.path.basename(f)

    try:
        x = int(base.split("_")[0][1:])
        y = int(base.split("_")[1][1:])
    except Exception:
        continue

    ms = base.split("_")[-1].replace(".npz", "")
    data = np.load(f, allow_pickle=True)

    if "neuron_ids" not in data or "responses" not in data:
        continue

    neuron_ids = data["neuron_ids"].astype(str)
    responses = data["responses"]

    stim_dict = responses_by_ms.setdefault(ms, {}).setdefault((x, y), {})
    for nid, resp in zip(neuron_ids, responses):
        stim_dict[nid] = resp

neurons_by_type = {t: [] for t in TARGET_TYPES}

for ms, stim_dict in responses_by_ms.items():
    neuron_set = set()
    for neuron_resp_dict in stim_dict.values():
        neuron_set.update(neuron_resp_dict.keys())
    for t in TARGET_TYPES:
        neurons_by_type[t].extend([
            nid for nid in neuron_set
            if cell_type_df["primary_type"].get(nid, "Unknown") == t
        ])

for t in TARGET_TYPES:
    neurons_by_type[t] = list(set(neurons_by_type[t]))


def generate_rv_curve(tau, r_start_px=4, r_max_px=20, frame_rate=1000,
                      gray_start_sec=0.6, gray_end_sec=0.5, ppd=1.0):

    angle_max_rad = np.deg2rad(r_max_px / ppd)
    angle_start_rad = np.deg2rad(r_start_px / ppd)

    t_min = tau / np.tan(angle_max_rad)
    t_max = tau / np.tan(angle_start_rad)

    looming_duration_sec = t_max - t_min
    print(tau,looming_duration_sec)
    looming_frames = int(np.ceil(looming_duration_sec * frame_rate))

    gray_start_frames = int(gray_start_sec * frame_rate)
    gray_end_frames = int(gray_end_sec * frame_rate)
    total_frames = gray_start_frames + looming_frames + gray_end_frames

    rv_frames = np.zeros(total_frames)

    rv_frames[:gray_start_frames] = 0.0

    for f in range(looming_frames):
        t_elapsed = f / frame_rate
        t_remain = t_max - t_elapsed
        if t_remain < 1e-6: t_remain = 1e-6
        angle_rad = np.arctan(tau / t_remain)
        rv_frames[gray_start_frames + f] = np.rad2deg(angle_rad)

    rv_frames[gray_start_frames + looming_frames:] = np.rad2deg(angle_max_rad)

    if len(rv_frames) > 500:
        rv_plot = rv_frames[500::10]
    else:
        rv_plot = rv_frames[::10]

    return rv_plot

def plot_max_peak_traces_with_rv(responses_by_ms, neurons_by_type):
    type_colors = {
        "DNp01": "#2F6DB3",  # blue
        "DNp02": "#F28E2B",  # orange
        "DNp03": "#D23B3B",  # red
        "DNp04": "#3DA44D",  # green
        "DNp11": "#7A60A8",  # purple
    }

    ms_list = sorted(responses_by_ms.keys())
    side_order = ["left", "right"]

    n_types = len(TARGET_TYPES)
    n_cols = len(ms_list) * len(side_order)

    fig, axes = plt.subplots(
        nrows=n_types,
        ncols=n_cols,
        figsize=(n_cols*2, n_types*1.5),
        constrained_layout=True,
        sharex=True
    )

    if n_types == 1:
        axes = np.expand_dims(axes, 0)
    if n_cols == 1:
        axes = axes[:, np.newaxis]

    for i, neuron_type in enumerate(TARGET_TYPES):
        ax_row = axes[i]
        ax_row[0].text(-0.3, 0.5, neuron_type, transform=ax_row[0].transAxes, va='center', ha='right')

        for j, ms in enumerate(ms_list):
            ms_num = ms.replace("ms", "")
            stim_dict = responses_by_ms[ms]
            tau_val = float(ms_num)/1000

            rv_curve = generate_rv_curve(tau_val)

            for k, side in enumerate(side_order):
                col_idx = j*2 + k
                ax = ax_row[col_idx]
                ax.axis("off")

                neurons = [
                    nid for nid in neurons_by_type[neuron_type]
                    if classification_df["side"].get(nid, "Unknown") == side
                ]
                if not neurons:
                    continue

                max_resp = None
                max_val = -np.inf
                for (x, y), neuron_resp_dict in stim_dict.items():
                    for nid in neurons:
                        if nid not in neuron_resp_dict:
                            continue
                        resp = neuron_resp_dict[nid]
                        peak = np.max(resp)
                        if peak > max_val:
                            max_val = peak
                            max_resp = resp

                if max_resp is not None:
                    color = type_colors.get(neuron_type, "#333333")
                    ax.plot(max_resp, color=color, linewidth=2.0)

                    rv_plot_scaled = rv_curve * max_val / np.max(rv_curve)
                    if len(rv_plot_scaled) < len(max_resp):
                        rv_plot_scaled = np.pad(rv_plot_scaled, (0, len(max_resp)-len(rv_plot_scaled)), 'edge')
                    else:
                        rv_plot_scaled = rv_plot_scaled[:len(max_resp)]
                    ax.plot(rv_plot_scaled, color='black', linestyle='--', linewidth=1.5)

                    ax.set_ylim(-max_val*1.05, max_val*1.05)
                    ax.set_xlim(0, len(max_resp)-1)

                    if neuron_type == "DNp03" and side == "right":
                        ax.text(0.2, 0.9, 'x', color='red',
                                transform=ax.transAxes, fontsize=20)

    for j, ms in enumerate(ms_list):
        for k, side in enumerate(side_order):
            col_idx = j*2 + k
            if side=='right':
                axes[0][col_idx].set_title(f"{ms}_L")
            else:
                axes[0][col_idx].set_title(f"{ms}_R")
            
    fname = os.path.join(peak_trace_root, "max_peak_traces_labeled_rv_generated.png")
    fig.savefig(fname, dpi=300, bbox_inches="tight")
    plt.close(fig)
    gc.collect()

plot_max_peak_traces_with_rv(responses_by_ms, neurons_by_type)
